Essential - start here!

In [ ]:
# bedpe file is the output of HiC_cluster3, either from clustering or cluster match 
bedpe_file = '~/example_data/combined-clusters.txt'
bedpe_skiprows = 0
grouping_col = 'GROUP'

out_dir = '~/example_data/cluster3'

###################################################################################

import pandas as pd
import os
import subprocess
from copy import deepcopy
import bioframe as bf

from scipy.stats import chi2_contingency
import pyBigWig
import statistics

import bedpe_analysis
from cluster_tools import sort_by_strings
from cooltools_called import mcool_pileup
from chromHMM_heatmap import heatmap_plot
from deepTools_pipeline import bed_pileup
import plotting
import statistics_functions

os.makedirs(out_dir,exist_ok=True)
bedpe_df = pd.read_csv(bedpe_file,sep='\t',header=0,skiprows=bedpe_skiprows)

if grouping_col == 'GROUP': bedpe_df_sorted = sort_by_strings(df=bedpe_df,sort_col='GROUP',order=['clust1','clust2','clust3','clust4','clust5','clust6'])
else: bedpe_df_sorted = bedpe_df.copy()

group_dict = {}
for group in bedpe_df_sorted[grouping_col].unique(): group_dict[group] = bedpe_df_sorted.loc[bedpe_df_sorted[grouping_col] == group,['chr1','x1','x2','chr2','y1','y2']].drop_duplicates(ignore_index=True)


Loop size

Determines the sizes of the loops in the different clusters, makes figures, and gets statistics.

In [ ]:
###################################################################################
bedpe_analysis.loop_size(out_dir=out_dir,bedpe_dict=group_dict,logY=False)

Loop classification

Classifies loops in each cluster as structural, CRE, or mixed based on association with CTCF & RAD21, enhancers, and genes

In [ ]:
CTCF_path = '~/example_data/CTCF_R1_TP461_S14_TP460_S13_0.01_peaks.narrowPeak'
RAD21_path = '~/example_data/RAD21_DMSO_hiConf.bed'
enhancer_path = '~/example_data/rep1_rep2_merge_0_AllStitched.table.txt'
promoter_prox = '~/example_data/gencode.v25.annotation_pp.bed'

###################################################################################

loop_dict = deepcopy(group_dict)

classifier_dict = {'CTCF':pd.read_csv(CTCF_path,sep='\t',header=None,names=['chr','start','end'],usecols=[0,1,2]),
                    'RAD21':pd.read_csv(RAD21_path,sep='\t',header=None,names=['chr','start','end'],usecols=[0,1,2]),
                    'enhancer':pd.read_csv(enhancer_path,sep='\t',header=None,names=['chr','start','end'],usecols=[1,2,3],skiprows=6),
                    'promoter':pd.read_csv(promoter_prox,sep='\t',header=None,names=['chr','start','end'],usecols=[0,1,2])
                    }
out_loop_dict = deepcopy(loop_dict)

classified_df = pd.DataFrame()
for loop_group,loop_df in loop_dict.items():
    anchor_mod = loop_df.copy()

    for anchor in [['chr1','x1','x2'],['chr2','y1','y2']]:
        for classifier,class_df in classifier_dict.items():
            overlap_df = bf.overlap(df1=loop_df[anchor],df2=class_df,cols1=anchor,cols2=['chr','start','end'])
            overlap_df.loc[overlap_df['start_'].notnull(),anchor[0] + '_' + classifier] = 1
            overlap_df.loc[overlap_df['start_'].isnull(),anchor[0] + '_' + classifier] = 0
            overlap_df = overlap_df[anchor + [anchor[0] + '_' + classifier]].sort_values(by=anchor[0] + '_' + classifier,ascending=False).drop_duplicates(subset=anchor)
            loop_df = loop_df.merge(overlap_df,on=anchor,how='left')
    
    loop_df['chr1_EorP'] = loop_df[['chr1_promoter','chr1_enhancer']].max(axis=1)
    loop_df['chr2_EorP'] = loop_df[['chr2_promoter','chr2_enhancer']].max(axis=1)

    for classifier in ['EorP','RAD21','CTCF']: loop_df[classifier] = loop_df[['chr1_' + classifier,'chr2_' + classifier]].sum(axis=1)
    loop_df = loop_df[['chr1','x1','x2','chr2','y1','y2','EorP','RAD21','CTCF']]

    loop_df.loc[(loop_df['CTCF'] == 2) & (loop_df['RAD21'] == 2) & (loop_df['EorP'] < 2),'classification'] = 'structural'
    loop_df.loc[((loop_df['CTCF'] < 2) | (loop_df['RAD21'] < 2)) & (loop_df['EorP'] == 2),'classification'] = 'CRE'
    loop_df.loc[(loop_df['CTCF'] == 2) & (loop_df['RAD21'] == 2) & (loop_df['EorP'] == 2),'classification'] = 'mixed'
    loop_df.loc[loop_df['classification'].isnull(),'classification'] = 'unclassified'

    for classifier in ['structural','CRE','mixed','unclassified']: classified_df.loc[classifier,loop_group] = loop_df[loop_df['classification'] == classifier].shape[0]

print(chi2_contingency(classified_df))

classified_df = 100 * classified_df/classified_df.sum()

plotting.stacked(count_table=classified_df.transpose(),title='',out_dir=out_dir,out_name='loop_classification',measure='loop proportion')


ChIP RPKM in anchors

Given a dictionary of RPKM-normalized BigWig files from ChIP-seq or CUT&RUN, this will identify the mean RPKM within RAD21 peaks that overlap the loop anchors in the different clusters. This RPKM is then normalized to RAD21 levels in the same regions.

In [ ]:
bed_intersect = '~/example_data/RAD21_DMSO_hiConf.bed'

# RPKM gets normalized to RAD21, so that needs to be included
bigWig_dict = {'D7_STAG1':'~/example_data/TP539_S10_b38d5.no_blacklisted.no_decoy_RPKM.bw',
               'RAD21':'~/example_data/TP557_S10_b38d5-only.no_blacklisted.no_decoy_lib.bw'}


###################################################################################

bed_intersect_df = pd.read_csv(bed_intersect,sep='\t',names=['chr','start','end'])

os.makedirs(out_dir + '/ChIP_intersect',exist_ok=True)

all_groups = pd.DataFrame()
for loop_group,loop_df in group_dict.items():
    loop_df['group'] = loop_group
    all_groups = pd.concat([all_groups,loop_df])

all_stats = pd.DataFrame()

for name,path in bigWig_dict.items():
    bw = pyBigWig.open(path)

    for anchor in [['chr1','x1','x2'],['chr2','y1','y2']]:
        anchor_subset = bf.overlap(df1=all_groups[anchor].drop_duplicates(subset=anchor),df2=bed_intersect_df,cols1=anchor,cols2=['chr','start','end'],how='inner')
        anchor_subset.reset_index(drop=True,inplace=True)

        for row in range(anchor_subset.shape[0]):
            vals = bw.values(anchor_subset.loc[row,'chr_'], anchor_subset.loc[row,'start_'], anchor_subset.loc[row,'end_'])
            anchor_subset.loc[row,name + '_' + anchor[0]] = statistics.mean(vals)
        anchor_subset = anchor_subset[anchor + [name + '_' + anchor[0]]].groupby(by=anchor,as_index=False).mean()
        all_groups = all_groups.merge(anchor_subset,on=anchor,how='left')
    all_groups[name] = all_groups[[name + '_chr1',name + '_chr2']].mean(axis=1)        
    bw.close()

all_groups = all_groups[['chr1','x1','x2','chr2','y1','y2','group'] + list(bigWig_dict.keys())]
all_groups[list(bigWig_dict.keys())] = all_groups[list(bigWig_dict.keys())].div(all_groups['RAD21'],axis=0)

for name in bigWig_dict.keys():
    if name == 'RAD21': continue
    data_names = []
    data_list = []
    for loop_group in group_dict.keys():
        data_names.append(name + '_' + loop_group)
        data_list.append(all_groups.loc[(all_groups['group'] == loop_group),name].dropna().tolist())
    stats_df = statistics_functions.kruskal_wilcoxon(data_names=data_names,data_list=data_list)
    all_stats = pd.concat([all_stats,stats_df])
all_stats.to_csv(out_dir + '/ChIP_intersect/anchor_ChIP.stats.txt',sep='\t',header=True,index=False)

melted_df = all_groups.melt(id_vars=['chr1','x1','x2','chr2','y1','y2','group'],value_vars=list(bigWig_dict.keys()),var_name='ChIP',value_name='signal')
melted_df.dropna(subset='signal',inplace=True)
melted_df.reset_index(drop=True,inplace=True)
plotting.box(melted_df=melted_df,xcol='group',ycol='signal',out_dir=out_dir + '/ChIP_intersect',out_name='anchor_ChIP_box',measure='mean anchor ChIP signal',title='',subplots=True,subplot_col='ChIP')

Cooltools pile-up analysis (bedpe)

Performs off-diagonal pile-up analysis for each of the clusters based on input mcool files.

In [ ]:
mcool_dict = {'DMSO_4hr': '',
            'dTAG_4hr':'',
            'dTAG_24hr':''}

split_diagonal = False
flank = 500000
resolution = 10000

###################################################################################

os.makedirs(out_dir + '/cooltools',exist_ok=True)

bedpe_dict = {}
for file,bedpe_df in group_dict.items():
    bedpe_dict[file] = bedpe_df.rename(columns={'chr1':'chrom1','x1':'start1','x2':'end1','chr2':'chrom2','y1':'start2','y2':'end2'})

mcool_pileup(mcool_dict=mcool_dict,bedpe_dict=bedpe_dict,out_dir=out_dir + '/cooltools',out_name='obs_exp_contacts',flank=flank,split_diagonal=split_diagonal,resolution=resolution,v_range=[-1,2])

Loop anchor gene annotation

Identifies genes whose promoters overlap the loop anchors. If an FPKM file is provided, it will also generate files and plots detailing the expression data.

In [ ]:
FPKM = pd.read_csv('~/example_data/nascentRNA_FPKM.txt',sep='\t',header=0)
temp_dir = '~/example_data/temp'

###################################################################################

use_cols = [i for i in FPKM.columns if 'DMSO_4hr' in i]
FPKM['FPKM'] = FPKM[use_cols].mean(axis=1)

FPKM = FPKM[['gene_id','gene_name','FPKM']]

os.makedirs(out_dir + '/annotation',exist_ok=True)
os.makedirs(temp_dir,exist_ok=True)
annotation = bedpe_analysis.bedtools_annotation(out_dir=out_dir + '/annotation',bedpe_dict=group_dict,FPKM_df=FPKM,temp_dir=temp_dir)

for group,df in annotation.items(): df.to_csv(out_dir + '/annotation/' + group + '_annotation.txt',sep='\t',header=True,index=False)

Loop strength scatter

Based on the count data in the cluster file, this will create joint plots for the two columns listed under samples

In [ ]:
samples = ['DMSO_merge','4hr_merge']

###################################################################################

os.makedirs(out_dir,exist_ok=True)

bedpe_df_sorted_plus = bedpe_df_sorted.loc[(bedpe_df_sorted['GROUP'] == 'clust4')].copy()

bedpe_df_sorted_plus[samples] = bedpe_df_sorted_plus[samples] + 0.0001

palette = {'FC > 2':'tab:blue','FC < 2':'darkgrey'}

palette = {'clust1':'black','clust2':'red','clust3':'orange','clust4':'blue','clust5':'green','clust6':'grey'}
plotting.joint(melted_df=bedpe_df_sorted_plus,ycol=samples[1],xcol=samples[0],ycol_measure='balanced counts + 0.0001',xcol_measure='balanced counts + 0.0001',out_dir=out_dir,out_name=samples[0] + '_vs_' + samples[1],title=samples[0] + '_vs_' + samples[1],logX=True,logY=True,modify_labels=True,hue_col='GROUP',add_n='hue_col',palette=palette)


Loop strength box plot

Creates box plots of the count data from the cluster file

In [ ]:
###################################################################################
bedpe_df_sorted_plus = bedpe_df_sorted.copy()

samples = [i for i in bedpe_df_sorted_plus.columns if i not in ['GROUP','chr1','x1','x2','chr2','y1','y2']]
bedpe_df_sorted_plus[samples] = bedpe_df_sorted_plus[samples] + 0.0001

melted_df = bedpe_df_sorted_plus.melt(id_vars=['GROUP','chr1','x1','x2','chr2','y1','y2'],value_vars=samples,var_name='conditions',value_name='counts')


stats_summary = pd.DataFrame()
palette = ['darkgrey', 'forestgreen']
for clust in ['clust1','clust2','clust3','clust4','clust5','clust6']:
    plotting.box(melted_df=melted_df[melted_df['GROUP'] == clust],xcol='conditions',ycol='counts',measure='balanced counts + 0.0001',title='',out_dir=out_dir,out_name=clust + '_box_plot',logY=True,Y_range=[-3,0])

    data_names = []
    data_list = []
    for condition in samples:
        data_names.append(clust + '_' + condition)
        data_list.append(melted_df.loc[(melted_df['GROUP'] == clust) & (melted_df['conditions'] == condition),'counts'].tolist())

    stats_df = statistics_functions.kruskal_wilcoxon(data_names=data_names,data_list=data_list)

    stats_summary = pd.concat([stats_summary,stats_df])

stats_summary.to_csv(out_dir + '/counts_stats.txt',sep='\t',header=True,index=False)

Relationship with differential ChIP analysis

Using the output from diffBind, this will plot the proportions of differential peaks in each of the clusters

In [ ]:
diffbind_output = '~/example_data/RAD21_DMSO_vs_dTAG_4hr.txt'
FDR_cutoff = 0.05
FC_cutoff = 0

###################################################################################

os.makedirs(out_dir + '/ChIP_intersect',exist_ok=True)

diffbind = pd.read_csv(diffbind_output,sep='\t',header=0)
diffbind.loc[(diffbind['FDR'] < FDR_cutoff) & (diffbind['Fold'] < -FC_cutoff),'sig'] = 'decreased'
diffbind.loc[(diffbind['FDR'] < FDR_cutoff) & (diffbind['Fold'] > FC_cutoff),'sig'] = 'increased'
diffbind['sig'] = diffbind['sig'].fillna('non-sig.')

sig_summary = pd.DataFrame()
for group,df in group_dict.items():
    out_df = df.copy()
    out_df = bf.overlap(out_df,diffbind[['Chr','Start','End','Fold','sig']],cols1=['chr1','x1','x2'],cols2=['Chr','Start','End'],suffixes=("",'_chr1'))
    out_df = bf.overlap(out_df,diffbind[['Chr','Start','End','Fold','sig']],cols1=['chr2','y1','y2'],cols2=['Chr','Start','End'],suffixes=("",'_chr2'))
    out_df.drop(columns=['Chr_chr1','Start_chr1','End_chr1','Chr_chr2','Start_chr2','End_chr2'],inplace=True)
    melt_df = out_df.melt(id_vars=['chr1','x1','x2','chr2','y1','y2'],value_vars=['sig_chr1','sig_chr1'],var_name='anchor',value_name='sig')
    sorted_melt = sort_by_strings(df=melt_df,sort_col='sig',order=['non-sig.','decreased','increased'])
    sorted_melt.reset_index(drop=True,inplace=True)
    sorted_melt.drop_duplicates(subset=['chr1','x1','x2','chr2','y1','y2'],keep='first',ignore_index=True,inplace=True)
    sig_summary.loc['non-sig',group] = sorted_melt[sorted_melt['sig'] == 'non-sig.'].shape[0]
    sig_summary.loc['sig',group] = sorted_melt[(sorted_melt['sig'] == 'decreased') | (sorted_melt['sig'] == 'increased')].shape[0]
    sig_summary.loc['no peak',group] = sorted_melt[sorted_melt['sig'].isnull()].shape[0]

print(chi2_contingency(sig_summary))

sig_summary = 100* sig_summary/sig_summary.sum()
plotting.stacked(measure='proportion of loops',title='',out_dir=out_dir + '/ChIP_intersect',out_name='differential_binding',count_table=sig_summary.transpose())

Relationship with diffBind FC

Also using the output from diffBind, this will identify peaks that overlap the loop anchors and use the peak with the max fold-change, with the mean of this taken across the two anchors.

In [ ]:
diffbind_output = '~/example_data/RAD21_DMSO_vs_dTAG_4hr.txt'
FDR_cutoff = 0.05
FC_cutoff = 0

###################################################################################

os.makedirs(out_dir + '/ChIP_intersect',exist_ok=True)

diffbind = pd.read_csv(diffbind_output,sep='\t',header=0)
diffbind.loc[(diffbind['FDR'] < FDR_cutoff) & (diffbind['Fold'] < -FC_cutoff),'sig'] = 'decreased'
diffbind.loc[(diffbind['FDR'] < FDR_cutoff) & (diffbind['Fold'] > FC_cutoff),'sig'] = 'increased'
diffbind['sig'] = diffbind['sig'].fillna('non-sig.')

sig_summary = pd.DataFrame()
data_list = []
data_names = []
out_dict = {}
for group,df in group_dict.items():
    out_df = df.copy()
    chr1_df = bf.overlap(out_df,diffbind[['Chr','Start','End','Fold','sig']],cols1=['chr1','x1','x2'],cols2=['Chr','Start','End'],suffixes=("",'_chr1'))
    chr1_df = chr1_df[['chr1','x1','x2','chr2','y1','y2','Fold_chr1']].dropna(subset=['Fold_chr1']).groupby(by=['chr1','x1','x2','chr2','y1','y2'],as_index=False).max()

    chr2_df = bf.overlap(out_df,diffbind[['Chr','Start','End','Fold','sig']],cols1=['chr2','y1','y2'],cols2=['Chr','Start','End'],suffixes=("",'_chr2'))
    chr2_df = chr2_df[['chr1','x1','x2','chr2','y1','y2','Fold_chr2']].dropna(subset=['Fold_chr2']).groupby(by=['chr1','x1','x2','chr2','y1','y2'],as_index=False).max()

    out_df = out_df.merge(chr1_df,on=['chr1','x1','x2','chr2','y1','y2'],how='left')
    out_df = out_df.merge(chr2_df,on=['chr1','x1','x2','chr2','y1','y2'],how='left')

    out_df['mean_fold'] = out_df[['Fold_chr1','Fold_chr2']].mean(axis=1)
    out_df.dropna(subset='mean_fold',inplace=True)
    out_df['group'] = group

    sig_summary = pd.concat([sig_summary,out_df])
    data_names.append(group)
    data_list.append(out_df['mean_fold'].tolist())
sig_summary.reset_index(drop=True,inplace=True)
plotting.box(melted_df=sig_summary,xcol='group',ycol='mean_fold',title='',out_dir=out_dir + '/ChIP_intersect',out_name='ChIP_FC_dTAG',measure='-log2FC')

stats_df = statistics_functions.kruskal_wilcoxon(data_names=data_names,data_list=data_list)
print(stats_df)


ChIP meta-analysis

This uses deepTools to create heatmaps of different factors that intersect RAD21 peaks at the loop anchors in the different clusters.

In [ ]:
bed_intersect = '~/example_data/RAD21_DMSO_hiConf.bed'
blacklisted_regions = '~/example_data/hg38-blacklist.v2.bed'

bigWig_dict = {'H3K27me3':'~/example_data/TP715_b38d5.no_blacklisted.no_decoy_RPKM.bw'}
bam_dict = None

color_dict = None
vmax_groups = None
up_down = 5000
out_name = 'histone_mods'

###################################################################################

os.makedirs(out_dir + '/ChIP_intersect',exist_ok=True)
os.makedirs(out_dir + '/intersect_temp',exist_ok=True)

bed_dict = {}
coord_dict = {}
for analysis,analysis_df in group_dict.items():
    concat = pd.concat([analysis_df[['chr1','x1','x2']],analysis_df[['chr2','y1','y2']].rename(columns={'chr2':'chr1','y1':'x1','y2':'x2'})])

    concat.to_csv(out_dir + '/intersect_temp/' + analysis + '.bed',sep='\t',header=False,index=False)

    func = 'bedtools intersect -wa -a ' + bed_intersect + ' -b ' + out_dir + '/intersect_temp/' + analysis + '.bed > ' + out_dir + '/intersect_temp/' + analysis + '_intersect.bed'
    subprocess.run(func,shell=True)
    bed_dict[analysis] = out_dir + '/intersect_temp/' + analysis + '_intersect.bed'

bed_pileup(out_dir=out_dir + '/ChIP_intersect',bigWig_dict=bigWig_dict,bam_dict=bam_dict,bed_dict=bed_dict,up_down=up_down,vmax_groups=vmax_groups,out_name=out_name,blacklisted_regions=blacklisted_regions)


ChromHMM relationship

Uses output from ChromHMM to look at the chromatin state in the anchors and span of the clusters

In [ ]:
segment_path = '~/example_data/hTERT-RPE1_12_segments.bed'
rename_path = '~/example_data/12state_rename.txt'

###################################################################################

os.makedirs(out_dir + '/chromHMM/anchor_input',exist_ok=True)
os.makedirs(out_dir + '/chromHMM/span_input',exist_ok=True)
order_df = pd.DataFrame()

rep = 0
for group,df in group_dict.items():
    for coord in ['anchor','span']:
        bed_path = out_dir + '/chromHMM/' + coord + '_input/' + group + '.bed'
        if coord == 'anchor': out_df = pd.concat([df[['chr1','x1','x2']],df[['chr2','y1','y2']].rename(columns={'chr2':'chr1','y1':'x1','y2':'x2'})])
        if coord == 'span': out_df = df[['chr1','x1','y2']]
        out_df.to_csv(bed_path,sep='\t',header=False,index=False)
    order_df.loc[rep,0] = group + '.bed'
    rep += 1
order_df.to_csv(out_dir + '/chromHMM/coordlistfile.txt',sep='\t',header=False,index=False)

for coord in ['anchor','span']:
    func = 'java -mx4000M -jar ~/apps/ChromHMM/ChromHMM.jar OverlapEnrichment -noimage -m ' + rename_path + ' -uniformscale -noimage -f ' + out_dir + '/chromHMM/coordlistfile.txt -colfields 0,1,2 ' + segment_path + ' ' + out_dir + '/chromHMM/' + coord + '_input/ ' + out_dir + '/chromHMM/' + coord
    subprocess.run(func,shell=True)

    heatmap_plot(path=out_dir + '/chromHMM/' + coord + '.txt',normalize=False)


ChromHMM proportions

Just another way of looking at enrichment of different chromatin states from ChromHMM

In [ ]:
segment_path = '~/example_data/hTERT-RPE1_12_segments.bed'
rename_path = '~/example_data/12state_rename.txt'

###################################################################################

segment_bed = pd.read_csv(segment_path,sep='\t',names=['chr','start','end','state'])
rename_df = pd.read_csv(rename_path,sep='\t',names=['short','state'])
rename_df['sort_col'] = rename_df['short'].copy()
rename_df['sort_col'] = rename_df['sort_col'].str.replace('U','').astype(int)
rename_df.sort_values('sort_col',inplace=True)

bedpe_df_sorted_cp = bedpe_df_sorted.copy()
bedpe_df_sorted_cp['loop'] = bedpe_df_sorted_cp['chr1'] + '-' + bedpe_df_sorted_cp['x1'].astype(str) + '-' + bedpe_df_sorted_cp['y2'].astype(str)

cluster_df = pd.concat([bedpe_df_sorted_cp[['chr1','x1','x2','loop','GROUP']].rename({'chr1':'chr','x1':'start','x2':'end'},axis=1),bedpe_df_sorted_cp[['chr2','y1','y2','loop','GROUP']].rename({'chr2':'chr','y1':'start','y2':'end'},axis=1)],ignore_index=True)

overlap_df = bf.overlap(df1=segment_bed,df2=cluster_df,cols1=['chr','start','end'],cols2=['chr','start','end'],how='inner',return_overlap=True)
overlap_df['overlap_length'] = overlap_df['overlap_end'] - overlap_df['overlap_start']

# # select most prominent state that is not "Low signal"
overlap_df = overlap_df[(overlap_df['state'] != 'U12') | (overlap_df['state'] != 'U9')]

overlap_grp = overlap_df[['state','GROUP_','loop_','overlap_length']].groupby(by=['state','GROUP_','loop_'],as_index=False).sum()

max_overlap = overlap_df.sort_values(by='overlap_length',ascending=False).drop_duplicates(subset=['loop_'],ignore_index=True)

prop = pd.DataFrame()
for group in ['clust1','clust2','clust3','clust4','clust5','clust6']:
    subset = max_overlap[max_overlap['GROUP_'] == group].copy()

    cluster_groups = pd.DataFrame()
    for state in rename_df['short'].unique():
        if state == 'U12': continue
        if state == 'U9': continue
        out_state = rename_df.loc[rename_df['short'] == state,'state'].values[0]
        prop.loc[out_state,group] = subset.loc[subset['state'] == state,'overlap_length'].shape[0]
    
    prop[group] = 100 * prop[group]/prop[group].sum()

prop = prop.transpose()

palette = {'Active promoter':'tab:blue', 'Weak promoter':'tab:orange', 'Intermediate enhancer':'tab:green',
       'Strong enhancer':'tab:red', 'Weak/poised enhancer':'tab:purple', 'Transcriptional transition':'tab:brown',
       'Transcriptional elongation':'tab:pink', 'Insulator':'tab:gray', 'Intermediate promoter':'tab:olive',
       'Repressive/polycomb':'tab:cyan', 'Low signal':'lime', 'Heterochromatin':'black'}
#plotting.stacked(measure='proportion of loop anchors',title='',out_dir=out_dir,out_name='chromHMM_anchor_stacked',count_table=prop,palette=palette)

value_vars = prop.columns
prop.reset_index(drop=False,inplace=True)

melted = prop.melt(id_vars='index',value_vars=value_vars,var_name='state',value_name='proportion')

plotting.bar(melted_df=melted,xcol='index',ycol='proportion',out_dir=out_dir,hue_col='state',title='',palette=palette,out_name='chromHMM_anchor',measure='proportion of loop anchors')